In [3]:
import sys
sys.path.append("E:\\wikiart_project")

from config import (
    set_seed, get_split_indices, get_transforms,
    WikiArtDataset, LEARNING_RATE, LOSS_WEIGHTS,
    NUM_STYLES, NUM_GENRES, SAVE_DIR
)

import os
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models
from datasets import load_dataset
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

# ── Set seed ─────────────────────────────────────────────
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── Settings (3 epoch only, multi-task with SQRT class weights) ─
NUM_EPOCHS = 3
IMAGE_SIZE = 224
BATCH_SIZE = 32
MODEL_NAME = "resnet50_class_weighted_sqrt"

# ── Load dataset ────────────────────────────────────────
print("Loading dataset...")
dataset = load_dataset("huggan/wikiart", split="train")
splits = get_split_indices(len(dataset))
print(f"Train: {len(splits['train'])} | Val: {len(splits['val'])}")

# ── Compute class weights from training set (SQRT version) ─
print("Computing class weights (square-root inverse frequency)...")
style_counts = np.zeros(NUM_STYLES)
genre_counts = np.zeros(NUM_GENRES)

for idx in tqdm(splits['train'], desc="Counting classes"):
    sample = dataset[idx]
    style_counts[sample['style']] += 1
    genre_counts[sample['genre']] += 1

# Square-root inverse frequency: smoother than pure inverse frequency
style_weights = 1.0 / np.sqrt(style_counts + 1)
style_weights = style_weights / style_weights.sum() * NUM_STYLES

genre_weights = 1.0 / np.sqrt(genre_counts + 1)
genre_weights = genre_weights / genre_weights.sum() * NUM_GENRES

style_weights = torch.tensor(style_weights, dtype=torch.float32).to(device)
genre_weights = torch.tensor(genre_weights, dtype=torch.float32).to(device)
print("Class weights computed.")
print(f"Style weight range: {style_weights.min():.4f} to {style_weights.max():.4f}")
print(f"Genre weight range: {genre_weights.min():.4f} to {genre_weights.max():.4f}")

# ── DataLoaders ─────────────────────────────────────────
train_transform, eval_transform = get_transforms(IMAGE_SIZE)

train_dataset = WikiArtDataset(dataset, splits['train'], train_transform)
val_dataset   = WikiArtDataset(dataset, splits['val'],   eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# ── Multi-task ResNet-50 ─────────────────────────────────
class MultiTaskResNet(nn.Module):
    def __init__(self, num_styles=NUM_STYLES, num_genres=NUM_GENRES):
        super().__init__()
        backbone = models.resnet50(weights='IMAGENET1K_V1')
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])
        self.style_head = nn.Linear(2048, num_styles)
        self.genre_head = nn.Linear(2048, num_genres)

    def forward(self, x):
        f = self.backbone(x).flatten(1)
        return self.style_head(f), self.genre_head(f)

model = MultiTaskResNet().to(device)

# ── Weighted CrossEntropy Loss ──────────────────────────
style_criterion = nn.CrossEntropyLoss(weight=style_weights)
genre_criterion = nn.CrossEntropyLoss(weight=genre_weights)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scaler = torch.cuda.amp.GradScaler()

# ── Checkpoint resume ───────────────────────────────────
CHECKPOINT_PATH = f"{SAVE_DIR}\\{MODEL_NAME}_checkpoint.pth"
start_epoch = 0
train_losses, val_losses = [], []
train_accs,   val_accs   = [], []
val_macro_f1s = []

if os.path.exists(CHECKPOINT_PATH):
    print("Found checkpoint, resuming...")
    ckpt = torch.load(CHECKPOINT_PATH)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scaler.load_state_dict(ckpt['scaler_state'])
    start_epoch = ckpt['epoch'] + 1
    train_losses = ckpt['train_losses']
    val_losses   = ckpt['val_losses']
    train_accs   = ckpt['train_accs']
    val_accs     = ckpt['val_accs']
    val_macro_f1s = ckpt.get('val_macro_f1s', [])
    print(f"Resumed from epoch {start_epoch}")

print("Model ready!")

# ── Training loop ───────────────────────────────────────
for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, styles, genres in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]"):
        images, styles, genres = images.to(device), styles.to(device), genres.to(device)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            style_out, genre_out = model(images)
            loss = LOSS_WEIGHTS['style'] * style_criterion(style_out, styles) + \
                   LOSS_WEIGHTS['genre'] * genre_criterion(genre_out, genres)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        correct      += (style_out.argmax(1) == styles).sum().item()
        total        += styles.size(0)

    train_losses.append(running_loss / len(train_loader))
    train_accs.append(correct / total)

    # —— Validate ——
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    all_preds, all_true = [], []

    with torch.no_grad():
        for images, styles, genres in tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]"):
            images, styles, genres = images.to(device), styles.to(device), genres.to(device)
            with torch.cuda.amp.autocast():
                style_out, genre_out = model(images)
                loss = LOSS_WEIGHTS['style'] * style_criterion(style_out, styles) + \
                       LOSS_WEIGHTS['genre'] * genre_criterion(genre_out, genres)
            val_loss    += loss.item()
            val_correct += (style_out.argmax(1) == styles).sum().item()
            val_total   += styles.size(0)
            all_preds.extend(style_out.argmax(1).cpu().numpy())
            all_true.extend(styles.cpu().numpy())

    val_losses.append(val_loss / len(val_loader))
    val_accs.append(val_correct / val_total)
    macro_f1 = f1_score(all_true, all_preds, average='macro')
    val_macro_f1s.append(macro_f1)

    print(f"Epoch {epoch+1}: Train Acc={train_accs[-1]:.4f} | Val Acc={val_accs[-1]:.4f} | Val Macro-F1={macro_f1:.4f}")

    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scaler_state': scaler.state_dict(),
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accs': train_accs,
        'val_accs': val_accs,
        'val_macro_f1s': val_macro_f1s,
    }, CHECKPOINT_PATH)

torch.save(model.state_dict(), f"{SAVE_DIR}\\{MODEL_NAME}_wikiart.pth")
history = {'train_losses': train_losses, 'val_losses': val_losses,
           'train_accs': train_accs, 'val_accs': val_accs,
           'val_macro_f1s': val_macro_f1s}
with open(f"{SAVE_DIR}\\{MODEL_NAME}_history.json", "w") as f:
    json.dump(history, f)

print("\n" + "="*70)
print("ABLATION 2 - SQRT WEIGHTS RESULT")
print("="*70)
print(f"NO weights         (5 epoch baseline): Val Acc=0.5916, Macro-F1=0.5610")
print(f"INVERSE freq       (3 epoch):          Val Acc=0.5178, Macro-F1=0.4976")
print(f"SQRT inverse freq  (3 epoch):          Val Acc={val_accs[-1]:.4f}, Macro-F1={val_macro_f1s[-1]:.4f}")
print("All done!")

Using device: cuda
Loading dataset...


Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/45 [00:00<?, ?it/s]

Train: 57010 | Val: 12217
Computing class weights (square-root inverse frequency)...


Counting classes: 100%|██████████| 57010/57010 [09:31<00:00, 99.79it/s] 


...

Epoch 1: Train Acc=0.4627 | Val Acc=0.5440 | Val Macro-F1=0.5207


C:\Users\yimin\AppData\Local\Temp\ipykernel_22792\380145879.py:126: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 2/3 [Train]: 100%|██████████| 1782/1782 [26:12<00:00,  1.13it/s]
C:\Users\yimin\AppData\Local\Temp\ipykernel_22792\380145879.py:150: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 2/3 [Val]: 100%|██████████| 382/382 [05:19<00:00,  1.20it/s]


Epoch 2: Train Acc=0.5611 | Val Acc=0.5609 | Val Macro-F1=0.5420


C:\Users\yimin\AppData\Local\Temp\ipykernel_22792\380145879.py:126: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 3/3 [Train]: 100%|██████████| 1782/1782 [26:14<00:00,  1.13it/s]
C:\Users\yimin\AppData\Local\Temp\ipykernel_22792\380145879.py:150: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 3/3 [Val]: 100%|██████████| 382/382 [05:24<00:00,  1.18it/s]


Epoch 3: Train Acc=0.6219 | Val Acc=0.5856 | Val Macro-F1=0.5532

ABLATION 2 - SQRT WEIGHTS RESULT
NO weights         (5 epoch baseline): Val Acc=0.5916, Macro-F1=0.5610
INVERSE freq       (3 epoch):          Val Acc=0.5178, Macro-F1=0.4976
SQRT inverse freq  (3 epoch):          Val Acc=0.5856, Macro-F1=0.5532
All done!
